# Merge S9 — 5-fold ensemble @ 150 epoch (Dataset020)

Train **cả 5 fold (0–4)** với trainer **150 epoch** (run hoàn chỉnh, LR về 0 ở epoch 150) → ensemble (trung bình softmax) để +0.5–1.5% Dice ổn định.

**Quyết định:** dùng **150ep cho tất cả** để nhất quán (ensemble yêu cầu mọi fold cùng trainer/folder). Fold 250-dở (nếu có) **bị bỏ**, train lại ở 150.

**CÔ LẬP:** ghi vào trainer MỚI `nnUNetTrainer_150epochs__...` → **không đụng** `nnUNetTrainer_250epochs__.../fold_0` (d20 gốc vẫn nguyên). Pred ghi folder mới `/content/pred_ens150_*`.

**Mỗi session** (Colab reinstall/xoá `/content`): chạy lại **cell 2 (tạo trainer)** + **cell 3 (preprocessed local)** trước khi train/resume.


In [ ]:
!pip install -q nnunetv2


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 134.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 138.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 11.3 MB/s eta 0:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 0) Env (preprocessed LOCAL cho nhanh)


In [ ]:
import os
from pathlib import Path
os.environ["nnUNet_raw"]          = "/content/drive/MyDrive/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"          # LOCAL
os.environ["nnUNet_results"]      = "/content/drive/MyDrive/nnUNet_results"  # Drive: checkpoint persist
os.makedirs("/content/nnUNet_preprocessed", exist_ok=True)

RAW = Path("/content/drive/MyDrive/nnUNet_raw")
ZIB, IMO = RAW/"Dataset001_KneeOA", RAW/"Dataset012_iMorphics"
TR, PLANS = "nnUNetTrainer_150epochs", "nnUNetResEncUNetLPlans"
print("raw:", [d for d in os.listdir(os.environ["nnUNet_raw"]) if d.startswith("Dataset020")])


raw: ['Dataset020_KneeUnion']


## 1) Tạo trainer `nnUNetTrainer_150epochs` (CHẠY MỖI SESSION)
Clone file 250 → đổi số. Package bị reinstall mỗi session nên phải tạo lại.


In [ ]:
!pip install -q --force-reinstall --no-deps "numpy==2.1.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 137.8 MB/s eta 0:00:00


In [ ]:
import nnunetv2, os, re, glob
base = os.path.join(os.path.dirname(nnunetv2.__file__), "training/nnUNetTrainer")

# tim file nao thuc su chua class nnUNetTrainer_250epochs
target, src = None, None
for f in glob.glob(os.path.join(base, "**/*.py"), recursive=True):
    t = open(f).read()
    if re.search(r"class nnUNetTrainer_250epochs\b", t):
        target, src = f, t; break
print("File chua class 250:", target)

if "nnUNetTrainer_150epochs" not in src:
    block = re.search(r"class nnUNetTrainer_250epochs\b.*?(?=\nclass |\Z)", src, re.S).group(0).replace("250", "150")
    open(target, "a").write("\n\n" + block)
    print("Da them nnUNetTrainer_150epochs vao", target)
else:
    print("nnUNetTrainer_150epochs da co san")

# verify nnU-Net nhan dien
from nnunetv2.utilities.find_class_by_name import recursive_find_python_class
cls = recursive_find_python_class(base, "nnUNetTrainer_150epochs", "nnunetv2.training.nnUNetTrainer")
print("trainer 150 san sang:", cls)


File chua class 250: /usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/variants/training_length/nnUNetTrainer_Xepochs.py
Da them nnUNetTrainer_150epochs vao /usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/variants/training_length/nnUNetTrainer_Xepochs.py
trainer 150 san sang: <class 'nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs.nnUNetTrainer_150epochs'>


## 2) Preprocessed LOCAL (CHẠY MỖI SESSION nếu `/content` trống)
Nếu đã copy/preprocess trong session này thì bỏ qua. Chỉ `3d_fullres`, 8 process.


In [ ]:
import os
dst = "/content/nnUNet_preprocessed/Dataset020_KneeUnion/nnUNetResEncUNetLPlans.json"
if not os.path.exists(dst):
    print("Chua co preprocessed local -> chay plan_and_preprocess (chi 3d_fullres)...")
    get_ipython().system("nnUNetv2_plan_and_preprocess -d 20 -pl nnUNetPlannerResEncL -c 3d_fullres -np 8 --verify_dataset_integrity")
else:
    print("Da co preprocessed local, bo qua.")


Chua co preprocessed local -> chay plan_and_preprocess (chi 3d_fullres)...
Fingerprint extraction...
Dataset020_KneeUnion
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Extracting dataset fingerprint: 100% 544/544 [01:38<00:00,  5.52it/s]
Experiment planning...
Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [160. 382. 382.], 3d_lowres: [160, 382, 382]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 62, 'patch_size': (np.int64(384), np.int64(384)), 'median_image_size_in_voxels': array([382., 382.]), 'spacing': array([0.36458334, 0.36458334]), 'normalization_schemes': ['ZScoreNormalization'], 'u

## 3) Train 5 fold @150ep
Mỗi fold ~150×73s ≈ ~3h. Colab đứt → chạy lại **đúng lệnh + `--c`** (checkpoint ở Drive). Chạy tuần tự từng cell (hoặc gộp), theo dõi `Pseudo dice [8]`.


In [ ]:
!nnUNetv2_train 20 3d_fullres 0 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_150epochs --c


Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-07-10 18:09:08.681986: Using torch.compile...
2026-07-10 18:09:16.362642: do_dummy_2d_data_aug: False
2026-07-10 18:09:16.368227: Using splits from existing split file: /content/nnUNet_preprocessed/Dataset020_KneeUnion/splits_final.json
2026-07-10 18:09:16.371411: The split file contains 5 splits.
2026-07-10 18:09:16.373856: Desired fold for training: 0
2026-07-10 18:09:16.375905: This split has 435 training and 109 validation cases.
using pin_memory on device 0
using pin_memory on device 0

This is the configuration used by this training:
Configuration name: 3d_fu

In [ ]:
!nnUNetv2_train 20 3d_fullres 1 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_150epochs


Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-07-11 00:37:32.314093: Using torch.compile...
2026-07-11 00:37:33.492714: do_dummy_2d_data_aug: False
2026-07-11 00:37:33.497183: Using splits from existing split file: /content/nnUNet_preprocessed/Dataset020_KneeUnion/splits_final.json
2026-07-11 00:37:33.499874: The split file contains 5 splits.
2026-07-11 00:37:33.502145: Desired fold for training: 1
2026-07-11 00:37:33.504017: This split has 435 training and 109 validation cases.
using pin_memory on device 0
using pin_memory on device 0

This is the configuration used by this training:
Configuration name: 3d_fu

In [ ]:
!nnUNetv2_train 20 3d_fullres 2 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_150epochs --c


Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-07-11 14:52:28.451473: Using torch.compile...
2026-07-11 14:52:47.662807: do_dummy_2d_data_aug: False
2026-07-11 14:52:47.668298: Creating new 5-fold cross-validation split...
2026-07-11 14:52:47.675398: Desired fold for training: 2
2026-07-11 14:52:47.677879: This split has 435 training and 109 validation cases.
using pin_memory on device 0
using pin_memory on device 0

This is the configuration used by this training:
Configuration name: 3d_fullres
 {'data_identifier': 'nnUNetPlans_3d_fullres', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_si

In [ ]:
!nnUNetv2_train 20 3d_fullres 3 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_150epochs


Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-07-11 21:31:18.285565: Using torch.compile...
2026-07-11 21:31:19.536234: do_dummy_2d_data_aug: False
2026-07-11 21:31:19.541029: Using splits from existing split file: /content/nnUNet_preprocessed/Dataset020_KneeUnion/splits_final.json
2026-07-11 21:31:19.543799: The split file contains 5 splits.
2026-07-11 21:31:19.546197: Desired fold for training: 3
2026-07-11 21:31:19.548206: This split has 435 training and 109 validation cases.
using pin_memory on device 0
using pin_memory on device 0

This is the configuration used by this training:
Configuration name: 3d_fu

In [ ]:
!nnUNetv2_train 20 3d_fullres 4 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_150epochs --c


Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-07-12 17:08:48.415403: Using torch.compile...
2026-07-12 17:09:24.861561: do_dummy_2d_data_aug: False
2026-07-12 17:09:24.866982: Creating new 5-fold cross-validation split...
2026-07-12 17:09:24.873746: Desired fold for training: 4
2026-07-12 17:09:24.876231: This split has 436 training and 108 validation cases.
using pin_memory on device 0
using pin_memory on device 0

This is the configuration used by this training:
Configuration name: 3d_fullres
 {'data_identifier': 'nnUNetPlans_3d_fullres', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_si

## 4) Ensemble predict (5 fold) trên 2 held-out → folder MỚI
`-f 0 1 2 3 4` → nnU-Net tự trung bình softmax. Dùng `checkpoint_best.pth`.


In [ ]:
!nnUNetv2_predict -i {ZIB}/imagesTs -o /content/pred_ens150_zib_ts  -d 20 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_150epochs -f 0 1 2 3 4 -chk checkpoint_best.pth
!nnUNetv2_predict -i {IMO}/imagesTs -o /content/pred_ens150_imo_test -d 20 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_150epochs -f 0 1 2 3 4 -chk checkpoint_best.pth



#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 103 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 103 cases that I would like to predict

Predicting oaizib_405:
perform_everything_on_device: True
100% 8/8 [00:24<00:00,  3.06s/it]
100% 8/8 [00:06<00:00,  1.28it/s]
100% 8/8 [00:06<00:00,  1.27it/s]
100% 8/8 [00:06<00:00,  1.27it/s]
100% 8/8 [00:06<00:00,  1.27it/s]
sending off prediction to background worker for resampling and export
done with oaizib_405

Predicting oaizib_406:
perform_everything_on_device: True
100% 8/8 [00:06<00:00,  1.27it/s]
100% 8/8 [00:06<00:00

## 5) Eval ensemble vs d20 single-fold (Dice + ASSD + HD95, SimpleITK)


In [ ]:
import SimpleITK as sitk, numpy as np
from pathlib import Path
from collections import defaultdict

def surface_dists(gm, pm, sp):
    if gm.sum()==0 or pm.sum()==0: return np.nan, np.nan
    gi = sitk.GetImageFromArray(gm.astype(np.uint8)); gi.SetSpacing(sp)
    pi = sitk.GetImageFromArray(pm.astype(np.uint8)); pi.SetSpacing(sp)
    gc = sitk.GetArrayFromImage(sitk.LabelContour(gi, fullyConnected=True)).astype(bool)
    pc = sitk.GetArrayFromImage(sitk.LabelContour(pi, fullyConnected=True)).astype(bool)
    gd = sitk.GetArrayFromImage(sitk.Abs(sitk.SignedMaurerDistanceMap(gi, squaredDistance=False, useImageSpacing=True)))
    pd = sitk.GetArrayFromImage(sitk.Abs(sitk.SignedMaurerDistanceMap(pi, squaredDistance=False, useImageSpacing=True)))
    a, b = gd[pc], pd[gc]
    if a.size==0 or b.size==0: return np.nan, np.nan
    alld = np.concatenate([a, b]); return float(alld.mean()), float(np.percentile(alld, 95))

def eval_folder(gt_dir, pred_dir, classes):
    acc = defaultdict(lambda: {"d":[],"a":[],"h":[]})
    for gf in sorted(Path(gt_dir).glob("*.nii.gz")):
        pf = Path(pred_dir)/gf.name
        if not pf.exists(): continue
        g = sitk.ReadImage(str(gf)); p = sitk.ReadImage(str(pf)); sp = g.GetSpacing()
        ga = sitk.GetArrayFromImage(g); pa = sitk.GetArrayFromImage(p)
        for c in classes:
            gm, pm = ga==c, pa==c
            if gm.sum()==0: continue
            acc[c]["d"].append(2*(gm&pm).sum()/(gm.sum()+pm.sum()+1e-8))
            a, h = surface_dists(gm, pm, sp)
            if not np.isnan(a): acc[c]["a"].append(a); acc[c]["h"].append(h)
    return acc

NM = {2:"fem_cart",4:"med_tib",5:"lat_tib",6:"med_men",7:"lat_men",8:"patellar"}
REPORT = {"zib_ts":[2,4,5], "imo_test":[2,4,5,6,7,8]}
D20REF = {"zib_ts":{2:0.891,4:0.852,5:0.868},
          "imo_test":{2:0.868,4:0.851,5:0.902,6:0.859,7:0.900,8:0.859}}
def mn(x): return np.mean(x) if x else float("nan")

for key, gt, pred in [("zib_ts", ZIB/"labelsTs", "/content/pred_ens150_zib_ts"),
                      ("imo_test", IMO/"labelsTs", "/content/pred_ens150_imo_test")]:
    A = eval_folder(gt, pred, REPORT[key])
    print("\n" + "="*68 + f"\n{key}  |  5-fold ensemble @150ep  vs  d20 (fold0, 250ep)\n" + "="*68)
    print(f"{'class':11s}{'Dice':>8}{'ASSD':>7}{'HD95':>7}{'d20':>8}{'delta':>8}")
    for c in REPORT[key]:
        if c not in A or not A[c]["d"]: continue
        d = mn(A[c]["d"]); ref = D20REF[key].get(c)
        ds = f"{d-ref:+.3f}" if ref else "-"
        print(f"{NM[c]:11s}{d:8.3f}{mn(A[c]['a']):7.2f}{mn(A[c]['h']):7.2f}{(ref or 0):8.3f}{ds:>8}")
print("\n-> Ky vong ensemble: sun nhich +0.005~0.015 vs single-fold. Neu muon, chay them S8 postprocessing len /content/pred_ens150_*.")



zib_ts  |  5-fold ensemble @150ep  vs  d20 (fold0, 250ep)
class          Dice   ASSD   HD95     d20   delta
fem_cart      0.886   0.18   0.71   0.891  -0.005
med_tib       0.849   0.19   0.81   0.852  -0.003
lat_tib       0.866   0.21   0.87   0.868  -0.002

imo_test  |  5-fold ensemble @150ep  vs  d20 (fold0, 250ep)
class          Dice   ASSD   HD95     d20   delta
fem_cart      0.862   0.22   0.82   0.868  -0.006
med_tib       0.853   0.19   0.86   0.851  +0.002
lat_tib       0.897   0.16   0.79   0.902  -0.005
med_men       0.860   0.45   2.58   0.859  +0.001
lat_men       0.896   0.21   0.90   0.900  -0.004
patellar      0.855   0.27   1.35   0.859  -0.004

-> Ky vong ensemble: sun nhich +0.005~0.015 vs single-fold. Neu muon, chay them S8 postprocessing len /content/pred_ens150_*.


## Kết luận
- Nếu sụn (2,4,5) tăng vs d20 → 5-fold ensemble @150ep là deliverable mới (mạnh + ổn định hơn single-fold).
- Có thể chồng thêm **postprocessing (S8)** lên pred ensemble để dọn biên.
- d20 gốc (250ep fold0) vẫn nguyên trong folder trainer 250 → không mất gì.
